Write the data one row at a time by using CSV and "append" file writing mode, as demonstrated below.

In your real use case, you'd want to be making an API call, getting the data and then *writing the data to the CSV before* moving on to the next API call. 

You want to avoid making *all* of the API calls and collecting *all* of the data at once, because (1) that risks losing everything if the program crashes after running for a long time and (2) it will potentially strain your machine's memory.

Lastly you're giving yourself the ability to structure your code in a way that checks for the last stored record and can resume the scrape/API calling from the point that you things crashed (in the event of a bug).

In [1]:
import pandas as pd
import requests
import time
import re

In [2]:
# Setup
API_KEY = "yMgwnlBsql8axIBF9rnaf697f5tLlc9S2KYyO2P9"
BASE_URL = "https://commonchemistry.cas.org/api"
data_path = "data/CDPH Search results - 1_12_2026, 3_34_15 PM.csv"
data = pd.read_csv(data_path, low_memory = False)

# Function to extract CAS numbers that are already in the ingredient name
def get_cas_from_text(text):
    if not text.strip():
        return ""
    cas_numbers = re.findall(r'\b\d{2,7}-\d{2}-\d\b', str(text))
    return cas_numbers if cas_numbers else ""

# Function to clean ingredient name (remove CAS numbers)
def clean_name(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r'\d{2,7}-\d{2}-\d', '', text)  # Remove CAS
    text = re.sub(r'/\s*\d+[-\d/\s]*', '', text)   # Remove extra numbers
    return text.strip()


def get_official_name(unique_cas):
    if not unique_cas or unique_cas == "":
        return ""
    
    url = f"{BASE_URL}/detail"
    headers = {"X-Api-Key": API_KEY}
    params = {"cas_rn": unique_cas}
    
    try:
        response = requests.get(url, headers=headers, params=params)
        if response.status_code == 200:
            return response.json().get('name', 'NOT FOUND')
        else:
            return "NOT FOUND"
    except:
        return "ERROR"

In [3]:
#Find CAS numbers
unique_cas = data["Ingredient Name"].unique()
#pd.DataFrame(unique_cas, columns = ["Ingredient Name"]).to_csv("ingredients.csv", index = False)

In [4]:
errors = []
cas_metadata = {}
for ingredient in unique_cas:
        cas_metadata[ingredient] = {}

In [5]:
cas_metadata

{'Titanium dioxide (CI 77891) 13463-67-7 / 1317-70-0 / 1317-80-2 / 98084-96-9': {},
 'CITRONELLOL ': {},
 'LINALOOL ': {},
 'HEXYL CINNAMAL ': {},
 'Limonene (1-methyl-4-prop-1-en-2-yl-cyclohexene; dl-limonene (racemic); Dipentene; (R)-p-mentha-1,8-diene; (d-limonene); (S)-p-mentha-1,8-diene; (l-limonene)) 5989-27-5 / 138-86-3 / 7705-14-8 / 5989-54-8': {},
 'Lilial ': {},
 'Aloe vera, non-decolorized (unfiltered) whole leaf extract ': {},
 '1,6-Octadien-3-ol, 3,7-dimethyl- (LINALOOL) 78-70-6': {},
 'CITRAL ': {},
 'GERANIOL ': {},
 'Citronellol /+- 3,7-dimethyloct-6-en-1-ol (CITRONELLOL; (3R)-3,7-dimethyloct-6-en-1-ol; (3S)-3,7-dimethyloct-6-en-1-ol) 106-22-9 / 26489-01-0 / 1117-61-9 / 7540-51-4': {},
 'Benzyl benzoate 120-51-4': {},
 'BUTYLPHENYL METHYLPROPIONAL ': {},
 '2-Benzylideneoctanal (HEXYL CINNAMAL) 101-86-0': {},
 '2-Phenoxyethyl isobutyrate 103-60-6': {},
 '1,3,4,6,7,8-Hexahydro-4,6,6,7,8,8-hexamethylcyclopenta-?-2-benzopyran (Hexamethylindanopyran) 1222-05-5': {},
 'Isoeug

In [6]:
# GO THROUGH DICTIONARY and apply ur functions
for ingredient, empty_dict in cas_metadata.items():
        try:
            empty_dict.update({
                'cas_number': get_cas_from_text(ingredient),
                'clean_name': clean_name(ingredient),
                #'true_cas_number': get_official_name(ingredient)
            })
        except AttributeError:
            errors.append(ingredient)

In [8]:
cas_metadata

{'Titanium dioxide (CI 77891) 13463-67-7 / 1317-70-0 / 1317-80-2 / 98084-96-9': {'cas_number': ['13463-67-7',
   '1317-70-0',
   '1317-80-2',
   '98084-96-9'],
  'clean_name': 'Titanium dioxide (CI 77891)  /  /  /'},
 'CITRONELLOL ': {'cas_number': '', 'clean_name': 'CITRONELLOL'},
 'LINALOOL ': {'cas_number': '', 'clean_name': 'LINALOOL'},
 'HEXYL CINNAMAL ': {'cas_number': '', 'clean_name': 'HEXYL CINNAMAL'},
 'Limonene (1-methyl-4-prop-1-en-2-yl-cyclohexene; dl-limonene (racemic); Dipentene; (R)-p-mentha-1,8-diene; (d-limonene); (S)-p-mentha-1,8-diene; (l-limonene)) 5989-27-5 / 138-86-3 / 7705-14-8 / 5989-54-8': {'cas_number': ['5989-27-5',
   '138-86-3',
   '7705-14-8',
   '5989-54-8'],
  'clean_name': 'Limonene (1-methyl-4-prop-1-en-2-yl-cyclohexene; dl-limonene (racemic); Dipentene; (R)-p-mentha-1,8-diene; (d-limonene); (S)-p-mentha-1,8-diene; (l-limonene))  /  /  /'},
 'Lilial ': {'cas_number': '', 'clean_name': 'Lilial'},
 'Aloe vera, non-decolorized (unfiltered) whole leaf ext

In [7]:

# Step Extract CAS numbers and clean names
print("Step 1: Processing ingredient names...")
unique_cas['CAS_Number'] = unique_cas['Ingredient Name'].apply(get_cas_from_text)
unique_cas['Clean_Name'] = unique_cas['Ingredient Name'].apply(clean_name)

# Step Save results
output_file = 'data/CDPH_ingredients_standardized.csv'
unique_cas.to_csv(output_file, index=False)

print(f"\n✓ Done! Results saved to: {output_file}")
print(f"\nSummary:")
print(f"Total rows: {len(unique_cas)}")
print(f"Rows with CAS numbers: {(unique_cas['CAS_Number'] != '').sum()}")
print(f"Official names retrieved: {(unique_cas['Official_Name'] != '').sum()}")

Step 1: Processing ingredient names...


IndexError: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices

In [ ]:
# Step 2: Get official names from API (only for rows that have CAS numbers)
print("\nStep 2: Getting official names from API...")
data['Official_Name'] = ""

rows_with_cas = data[data['CAS_Number'] != '']
total = len(rows_with_cas)

for i, (index, row) in enumerate(rows_with_cas.iterrows(), 1):
    print(f"[{i}/{total}] Getting name for CAS: {row['CAS_Number']}")
    official_name = get_official_name(row['CAS_Number'])
    data.at[index, 'Official_Name'] = official_name
    time.sleep(0.5)  # Be nice to the API